In [1]:
import pandas as pd 
from pathlib import Path
from bs4 import BeautifulSoup
import json
 

In [2]:
test_file_path = Path("../fixtures/test1.html")

test_file_text = test_file_path.read_text(encoding="utf-8")

print("Total Characters on test1 webpage: ", len(test_file_text))

print("First 300 characters: \n", test_file_text[:300])


Total Characters on test1 webpage:  610582
First 300 characters: 
 <!DOCTYPE html>
<!-- saved from url=(0042)https://preppykitchen.com/french-macarons/ -->
<html lang="en-US"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><script type="text/javascript" async="" src="./How to Make Macarons (with 8 Troubleshooting Tips) - Preppy Kitchen_file


In [3]:
soup = BeautifulSoup(test_file_text, 'html.parser')

ld_blocks = soup.find_all('script', type = "application/ld+json" )
print("#of ld-blocks in test1:",len(ld_blocks))

#of ld-blocks in test1: 1


In [4]:
ld_flat = []
for block in ld_blocks:
  ld_flat.append(block.get_text())
ld_length = [len(block) for block in ld_flat]
print(ld_length) 

[21405]


In [5]:
ld_parsed = []
for block in ld_flat:
    parsed_block = json.loads(block)
    ld_parsed.append(parsed_block)
ld_parsed[0].keys()

dict_keys(['@context', '@graph'])

In [6]:
print(ld_parsed[0]['@context'])
print(type(ld_parsed[0]['@graph']), len(ld_parsed[0]['@graph']))

https://schema.org
<class 'list'> 11


In [7]:
for item in ld_parsed[0]['@graph']:
    if item.get('@type') == "Recipe":
        recipe = item
        print(recipe.keys())


dict_keys(['@type', 'name', 'author', 'description', 'datePublished', 'image', 'video', 'recipeYield', 'prepTime', 'cookTime', 'totalTime', 'recipeIngredient', 'recipeInstructions', 'aggregateRating', 'review', 'recipeCategory', 'recipeCuisine', 'keywords', 'nutrition', '@id', 'isPartOf', 'mainEntityOfPage'])


In [8]:
test_graph = [
    {"@type": "Article"},
    {"@type": ["WebPage", "FAQPage"]},
    {"@type": ["Recipe", "Article"]},
    {"name": "no type at all"},
    {"@type": "RecipeBook"}
]
recipe = None
for item in ld_parsed[0]['@graph']:
    item_type = item.get("@type", "none")
    if isinstance(item_type, str):
        match  = item_type == "Recipe"
        
    else:
        match = "Recipe" in item_type
        
    if match:
        recipe = item
print(recipe.keys())

dict_keys(['@type', 'name', 'author', 'description', 'datePublished', 'image', 'video', 'recipeYield', 'prepTime', 'cookTime', 'totalTime', 'recipeIngredient', 'recipeInstructions', 'aggregateRating', 'review', 'recipeCategory', 'recipeCuisine', 'keywords', 'nutrition', '@id', 'isPartOf', 'mainEntityOfPage'])


In [9]:
type(recipe["recipeInstructions"]), len(recipe["recipeInstructions"])

(list, 2)

In [10]:
recipe["recipeIngredient"]

['168 g blanched almond flour ((about 1½ cups))',
 '120 g powdered sugar ((about 1 cup))',
 '100 g egg whites (room temperature (from 3 large eggs))',
 '¼ teaspoon cream of tartar',
 '100 g granulated sugar ((½ cup))',
 '1 teaspoon vanilla extract',
 '1-2 drops food coloring ((optional))',
 '1 cup unsalted butter (softened (226g))',
 '5 large egg yolks (room temperature)',
 '½ cup granulated sugar ((100g))',
 '1 teaspoon vanilla extract',
 '3 tablespoons water',
 'Pinch of salt']

In [63]:
recipe["recipeInstructions"][0]['itemListElement']

[{'@type': 'HowToStep',
  'text': 'Line two baking sheets with parchment paper or silicone baking mats, and set aside.',
  'name': 'Line two baking sheets with parchment paper or silicone baking mats, and set aside.',
  'url': 'https://preppykitchen.com/french-macarons/#wprm-recipe-3336-step-0-0'},
 {'@type': 'HowToStep',
  'text': 'In a medium bowl, sift in the powdered sugar and almond flour (be sure not to press the mixture through the sieve just let it fall through). Transfer to the workbowl of a food processor and pulse for 20 seconds. Stop, scrape down the sides, and pulse for another 10 seconds. Sift the almond flour mixture again into the same medium bowl, and set aside.',
  'name': 'In a medium bowl, sift in the powdered sugar and almond flour (be sure not to press the mixture through the sieve just let it fall through). Transfer to the workbowl of a food processor and pulse for 20 seconds. Stop, scrape down the sides, and pulse for another 10 seconds. Sift the almond flour mi

In [92]:
first_section = recipe['recipeInstructions'][0]

all_steps = []
for sections in recipe["recipeInstructions"]:
    for step in sections['itemListElement']:
        all_steps.append(step['text'])
print(len(all_steps))

14
